In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tiennhat/dataset/dev.json
/kaggle/input/datasets/tiennhat/dataset/vocab.json
/kaggle/input/datasets/tiennhat/dataset/train.json
/kaggle/input/datasets/tiennhat/dataset/test.json
/kaggle/input/datasets/tiennhat/dataset/data_cleaned.json


In [2]:
import json
import re
from collections import Counter


VOCAB_PATH = "/kaggle/input/datasets/tiennhat/dataset/vocab.json"
DATA_PATH= "/kaggle/input/datasets/tiennhat/dataset/data_cleaned.json"


def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_vocab(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def tokenize_vi(text):
    """
    Tokenize đơn giản theo whitespace.
    Vì data của bạn hiện tại đã được normalize khoảng trắng.
    """
    return text.lower().split()


def calculate_coverage(data, vocab):
    total_tokens = 0
    covered_tokens = 0

    all_words = Counter()
    covered_words = Counter()
    missing_words = Counter()

    for item in data:
        vi = item.get("vi", "")

        words = tokenize_vi(vi)

        for word in words:
            total_tokens += 1
            all_words[word] += 1

            if word in vocab:
                covered_tokens += 1
                covered_words[word] += 1
            else:
                missing_words[word] += 1

    # Token coverage
    token_coverage = (
        covered_tokens / total_tokens * 100
        if total_tokens > 0 else 0
    )

    # Unique word coverage
    unique_words = set(all_words.keys())
    covered_unique = unique_words.intersection(vocab.keys())

    unique_coverage = (
        len(covered_unique) / len(unique_words) * 100
        if unique_words else 0
    )

    return {
        "total_tokens": total_tokens,
        "covered_tokens": covered_tokens,
        "token_coverage": token_coverage,

        "total_unique_words": len(unique_words),
        "covered_unique_words": len(covered_unique),
        "unique_coverage": unique_coverage,

        "all_words": all_words,
        "covered_words": covered_words,
        "missing_words": missing_words,
    }




In [3]:
data = load_data(DATA_PATH)
vocab = load_vocab(VOCAB_PATH)

result = calculate_coverage(data, vocab)


In [4]:
print("=" * 50)
print("VOCAB COVERAGE")
print("=" * 50)
print(f"Total sentences       : {len(data):,}")
print(f"Total tokens          : "
      f"{result['total_tokens']:,}")
print(f"Covered tokens        : "
      f"{result['covered_tokens']:,}")
print(f"Token coverage        : "
      f"{result['token_coverage']:.2f}%")
print(f"Unique Vietnamese words: "
      f"{result['total_unique_words']:,}")
print(f"Covered unique words   : "
      f"{result['covered_unique_words']:,}")
print(f"Unique word coverage   : "
      f"{result['unique_coverage']:.2f}%")


VOCAB COVERAGE
Total sentences       : 238,442
Total tokens          : 5,737,990
Covered tokens        : 5,551,302
Token coverage        : 96.75%
Unique Vietnamese words: 28,112
Covered unique words   : 5,275
Unique word coverage   : 18.76%


In [5]:
#note ve preprocessing, chua xoa punctuation, chua lowercase, neu can thiet co the test

In [7]:
# Cell 1 - Imports + paths

import json
import os
import re
import unicodedata
from collections import Counter

import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from model import VietHanBertConfig, VietHanBertModel


INPUT_DIR = "/kaggle/input/datasets/tiennhat/dataset"
WORK_DIR = "/kaggle/working/viet_han_bert"

TRAIN_PATH = os.path.join(INPUT_DIR, "train.json")
DEV_PATH = os.path.join(INPUT_DIR, "dev.json")
TEST_PATH = os.path.join(INPUT_DIR, "test.json")
OLD_VOCAB_PATH = os.path.join(INPUT_DIR, "vocab.json")

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, "vocab"), exist_ok=True)
os.makedirs(os.path.join(WORK_DIR, "checkpoints"), exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Train:", TRAIN_PATH)
print("Dev:", DEV_PATH)
print("Test:", TEST_PATH)

Device: cuda
Train: /kaggle/input/datasets/tiennhat/dataset/train.json
Dev: /kaggle/input/datasets/tiennhat/dataset/dev.json
Test: /kaggle/input/datasets/tiennhat/dataset/test.json


In [8]:
# Cell 2 - Load dataset

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(DEV_PATH, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

with open(OLD_VOCAB_PATH, "r", encoding="utf-8") as f:
    old_vocab = json.load(f)

print("Train:", len(train_data))
print("Dev:", len(dev_data))
print("Test:", len(test_data))
print("Old vocab:", len(old_vocab))
print(train_data[0])

Train: 236421
Dev: 1313
Test: 1293
Old vocab: 7430
{'vi': '" \' tiền của bạn hoặc cuộc sống của bạn , \' cô ấy nói một cách trìu mến .', 'cn': '比如 ， “ 要钱 还要 命 ” ， 她 深情 地说 .'}


In [9]:
# Cell 3 - Tokenization functions

def normalize_text(text):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize_vi(text):
    text = normalize_text(text)

    text = re.sub(
        r'([!"#$%&\'()*+,\-./:;<=>?@\[\]^_`{|}~])',
        r" \1 ",
        text
    )

    text = re.sub(
        r"([。！？；：，、（）「」『』“”‘’…])",
        r" \1 ",
        text
    )

    return text.split()


def tokenize_han(text):
    text = normalize_text(text)
    text = text.replace(" ", "")
    return list(text)


print(tokenize_vi("Tôi đã học tiếng Hán."))
print(tokenize_han("我 已 學 漢 。"))

['Tôi', 'đã', 'học', 'tiếng', 'Hán', '.']
['我', '已', '學', '漢', '。']


In [10]:
# Cell 4 - Build Vietnamese and Han vocabularies from train only

SPECIAL_VI = {"<PAD>": 0, "<UNK>": 1, "<CLS>": 2, "<SEP>": 3}

SPECIAL_HAN = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3}

vi_counter = Counter()
han_counter = Counter()

for item in train_data:
    vi_counter.update(tokenize_vi(item["vi"]))
    han_counter.update(tokenize_han(item["cn"]))

vocab_vi = dict(SPECIAL_VI)
vocab_han = dict(SPECIAL_HAN)

for token, _ in vi_counter.most_common():
    if token not in vocab_vi:
        vocab_vi[token] = len(vocab_vi)

for token, _ in han_counter.most_common():
    if token not in vocab_han:
        vocab_han[token] = len(vocab_han)

VI_VOCAB_PATH = os.path.join(WORK_DIR, "vocab", "vocab_vi.json")
HAN_VOCAB_PATH = os.path.join(WORK_DIR, "vocab", "vocab_han.json")

with open(VI_VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(vocab_vi, f, ensure_ascii=False, indent=2)

with open(HAN_VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(vocab_han, f, ensure_ascii=False, indent=2)

print("Vietnamese vocab:", len(vocab_vi))
print("Han vocab:", len(vocab_han))
print("Saved:", VI_VOCAB_PATH)
print("Saved:", HAN_VOCAB_PATH)

Vietnamese vocab: 31542
Han vocab: 5087
Saved: /kaggle/working/viet_han_bert/vocab/vocab_vi.json
Saved: /kaggle/working/viet_han_bert/vocab/vocab_han.json


In [11]:
# Cell 5 - VietHanTokenizer

class VietHanTokenizer:

    def __init__(self, vi_vocab_path, han_vocab_path):
        with open(vi_vocab_path, "r", encoding="utf-8") as f:
            self.vi_vocab = json.load(f)

        with open(han_vocab_path, "r", encoding="utf-8") as f:
            self.han_vocab = json.load(f)

        self.vi_id2token = {int(v): k for k, v in self.vi_vocab.items()}
        self.han_id2token = {int(v): k for k, v in self.han_vocab.items()}

        self.vi_pad_id = self.vi_vocab["<PAD>"]
        self.vi_unk_id = self.vi_vocab["<UNK>"]
        self.vi_cls_id = self.vi_vocab["<CLS>"]
        self.vi_sep_id = self.vi_vocab["<SEP>"]

        self.han_pad_id = self.han_vocab["<PAD>"]
        self.han_unk_id = self.han_vocab["<UNK>"]
        self.han_bos_id = self.han_vocab["<BOS>"]
        self.han_eos_id = self.han_vocab["<EOS>"]

    @staticmethod
    def normalize_text(text):
        text = unicodedata.normalize("NFC", text)
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    def tokenize_vi(self, text):
        text = self.normalize_text(text)

        text = re.sub(
            r'([!"#$%&\'()*+,\-./:;<=>?@\[\]^_`{|}~])',
            r" \1 ",
            text
        )

        text = re.sub(
            r"([。！？；：，、（）「」『』“”‘’…])",
            r" \1 ",
            text
        )

        return text.split()

    def tokenize_han(self, text):
        text = self.normalize_text(text)
        text = text.replace(" ", "")
        return list(text)

    def encode_vi(self, text, add_special_tokens=True):
        tokens = self.tokenize_vi(text)
        ids = [self.vi_vocab.get(token, self.vi_unk_id) for token in tokens]

        if add_special_tokens:
            ids = [self.vi_cls_id] + ids + [self.vi_sep_id]

        return ids

    def decode_vi(self, ids):
        tokens = []

        for idx in ids:
            idx = int(idx)

            if idx in {self.vi_pad_id, self.vi_cls_id, self.vi_sep_id}:
                continue

            tokens.append(self.vi_id2token.get(idx, "<UNK>"))

        text = " ".join(tokens)
        text = re.sub(r"\s+([,.!?;:])", r"\1", text)
        text = re.sub(r"\s+([。！？；：，、）】」』])", r"\1", text)
        text = re.sub(r"([（【「『])\s+", r"\1", text)

        return text

    def encode_han(self, text, add_bos=True, add_eos=True):
        tokens = self.tokenize_han(text)
        ids = [self.han_vocab.get(token, self.han_unk_id) for token in tokens]

        if add_bos:
            ids.insert(0, self.han_bos_id)

        if add_eos:
            ids.append(self.han_eos_id)

        return ids

    def decode_han(self, ids, skip_special_tokens=True):
        special_ids = {
            self.han_pad_id,
            self.han_bos_id,
            self.han_eos_id
        }
    
        tokens = []
    
        for idx in ids:
            idx = int(idx)
    
            if skip_special_tokens and idx in special_ids:
                continue
    
            tokens.append(self.han_id2token.get(idx, "<UNK>"))
    
        return "".join(tokens)

    @property
    def vi_vocab_size(self):
        return len(self.vi_vocab)

    @property
    def han_vocab_size(self):
        return len(self.han_vocab)


tokenizer = VietHanTokenizer(VI_VOCAB_PATH, HAN_VOCAB_PATH)

print("VI vocab:", tokenizer.vi_vocab_size)
print("Han vocab:", tokenizer.han_vocab_size)

VI vocab: 31542
Han vocab: 5087


In [12]:
# Cell 6 - Test tokenizer

test_vi = "Tôi đã học tiếng Hán."
test_han = "我 已 學 漢 。"

print("VI tokens:", tokenizer.tokenize_vi(test_vi))
print("VI ids:", tokenizer.encode_vi(test_vi))

han_ids = tokenizer.encode_han(test_han)
print("Han tokens:", tokenizer.tokenize_han(test_han))
print("Han ids:", han_ids)
print("Han decoded:", tokenizer.decode_han(han_ids))

VI tokens: ['Tôi', 'đã', 'học', 'tiếng', 'Hán', '.']
VI ids: [2, 229, 24, 65, 205, 5950, 5, 3]
Han tokens: ['我', '已', '學', '漢', '。']
Han ids: [2, 6, 189, 5013, 1, 7, 3]
Han decoded: 我已學<UNK>。


In [13]:
# Cell 7 - Dataset

class VietHanDataset(Dataset):

    def __init__(self, data, tokenizer, max_source_length=128, max_target_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_source_length = max_source_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        source_ids = self.tokenizer.encode_vi(item["vi"], add_special_tokens=True)
        target_ids = self.tokenizer.encode_han(item["cn"], add_bos=True, add_eos=True)

        source_ids = source_ids[:self.max_source_length]
        target_ids = target_ids[:self.max_target_length]

        decoder_input_ids = target_ids[:-1]
        labels = target_ids[1:]

        return {
            "input_ids": torch.tensor(source_ids, dtype=torch.long),
            "decoder_input_ids": torch.tensor(decoder_input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long)
        }

In [14]:
# Cell 8 - Collator

class VietHanCollator:

    def __init__(self, tokenizer):
        self.vi_pad_id = tokenizer.vi_pad_id
        self.han_pad_id = tokenizer.han_pad_id

    def __call__(self, batch):
        max_src_len = max(len(item["input_ids"]) for item in batch)
        max_tgt_len = max(len(item["decoder_input_ids"]) for item in batch)

        input_ids = []
        attention_mask = []
        decoder_input_ids = []
        labels = []

        for item in batch:
            src = item["input_ids"]
            tgt = item["decoder_input_ids"]
            label = item["labels"]

            src_pad_len = max_src_len - len(src)
            tgt_pad_len = max_tgt_len - len(tgt)
            label_pad_len = max_tgt_len - len(label)

            input_ids.append(torch.cat([src, torch.full((src_pad_len,), self.vi_pad_id, dtype=torch.long)]))

            attention_mask.append(torch.tensor([1] * len(src) + [0] * src_pad_len, dtype=torch.long))

            decoder_input_ids.append(torch.cat([tgt,torch.full((tgt_pad_len,), self.han_pad_id, dtype=torch.long)]))

            labels.append(torch.cat([label, torch.full((label_pad_len,), -100, dtype=torch.long)]))

        return {
            "input_ids": torch.stack(input_ids),
            "attention_mask": torch.stack(attention_mask),
            "decoder_input_ids": torch.stack(decoder_input_ids),
            "labels": torch.stack(labels)
        }

In [15]:
# Cell 9 - Create datasets and loaders

MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

train_dataset = VietHanDataset(
    train_data,
    tokenizer,
    max_source_length=MAX_SOURCE_LENGTH,
    max_target_length=MAX_TARGET_LENGTH
)

dev_dataset = VietHanDataset(
    dev_data,
    tokenizer,
    max_source_length=MAX_SOURCE_LENGTH,
    max_target_length=MAX_TARGET_LENGTH
)

test_dataset = VietHanDataset(
    test_data,
    tokenizer,
    max_source_length=MAX_SOURCE_LENGTH,
    max_target_length=MAX_TARGET_LENGTH
)

collator = VietHanCollator(tokenizer)

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=2,
    pin_memory=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=2,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Dev batches:", len(dev_loader))
print("Test batches:", len(test_loader))

Train batches: 14777
Dev batches: 83
Test batches: 81


In [16]:
# Cell 10 - Inspect one batch

batch = next(iter(train_loader))

for key, value in batch.items():
    print(key, value.shape)
    print(value[0])

input_ids torch.Size([16, 64])
tensor([   2,    7,   35, 1384,   57,  127,   10,  132, 1291,   13, 4505, 3273,
           5,    3,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0])
attention_mask torch.Size([16, 64])
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
decoder_input_ids torch.Size([16, 83])
tensor([   2,    6,  145,   22,  945, 1135, 1938,   56,    4,   10,  861,  235,
          76,   23,  208, 1274,    7,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
    

In [19]:
# Cell 13 - Build model

config = VietHanBertConfig(
    vocab_size=tokenizer.vi_vocab_size,
    hidden_size=512,
    num_hidden_layers=6,
    num_attention_heads=8,
    intermediate_size=2048,
    max_position_embeddings=MAX_SOURCE_LENGTH,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
    han_vocab_size=tokenizer.han_vocab_size,
    decoder_layers=4,
    decoder_heads=8,
    decoder_ffn_dim=2048,
    max_target_position_embeddings=MAX_TARGET_LENGTH
)

model = VietHanBertModel(config).to(DEVICE)

num_params = sum(
    param.numel()
    for param in model.parameters()
)

print("Parameters:", f"{num_params:,}")

Parameters: 57,223,168


In [20]:
#cell 14

batch = next(iter(train_loader))

batch = {
    key: value.to(DEVICE)
    for key, value in batch.items()
}

model.eval()

with torch.no_grad():
    outputs = model(**batch)

print("Loss:", outputs["loss"].item())
print("Logits:", outputs["logits"].shape)

Loss: 8.644762992858887
Logits: torch.Size([16, 127, 5087])


In [21]:
# Cell 15 - Overfit subset of 100 samples

subset_size = 100

small_train_data = train_data[:subset_size]

small_dataset = VietHanDataset(
    small_train_data,
    tokenizer,
    max_source_length=MAX_SOURCE_LENGTH,
    max_target_length=MAX_TARGET_LENGTH
)

small_loader = DataLoader(
    small_dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=collator
)

small_model = VietHanBertModel(config).to(DEVICE)

optimizer = AdamW(
    small_model.parameters(),
    lr=1e-4,
    weight_decay=0.0
)

small_model.train()

for epoch in range(20):
    total_loss = 0.0

    for batch in small_loader:
        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        outputs = small_model(**batch)
        loss = outputs["loss"]

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(small_model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(small_loader)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Loss: {avg_loss:.4f}"
    )

Epoch 01 | Loss: 7.9254
Epoch 02 | Loss: 7.0678
Epoch 03 | Loss: 6.4161
Epoch 04 | Loss: 5.8887
Epoch 05 | Loss: 5.4382
Epoch 06 | Loss: 5.0544
Epoch 07 | Loss: 4.6969
Epoch 08 | Loss: 4.3826
Epoch 09 | Loss: 4.0654
Epoch 10 | Loss: 3.7731
Epoch 11 | Loss: 3.4773
Epoch 12 | Loss: 3.2119
Epoch 13 | Loss: 2.9433
Epoch 14 | Loss: 2.7027
Epoch 15 | Loss: 2.4673
Epoch 16 | Loss: 2.2146
Epoch 17 | Loss: 2.0140
Epoch 18 | Loss: 1.8042
Epoch 19 | Loss: 1.5727
Epoch 20 | Loss: 1.4156


In [22]:
# Cell 16 - Training config

EPOCHS = 5
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM_STEPS = 2

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [23]:
# Cell 17 - Validation loss

def evaluate_loss(model, loader, device):
    model.eval()

    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for batch in loader:
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            outputs = model(**batch)

            total_loss += outputs["loss"].item()
            total_batches += 1

    model.train()

    return total_loss / total_batches

In [24]:
# Cell 18 - Full training

scaler = torch.cuda.amp.GradScaler(
    enabled=torch.cuda.is_available()
)

model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader):
        batch = {
            key: value.to(DEVICE)
            for key, value in batch.items()
        }

        with torch.cuda.amp.autocast(
            enabled=torch.cuda.is_available()
        ):
            outputs = model(**batch)
            loss = outputs["loss"]
            scaled_loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(scaled_loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()

        if step % 100 == 0:
            print(
                f"Epoch {epoch + 1} | "
                f"Step {step}/{len(train_loader)} | "
                f"Loss {loss.item():.4f}"
            )

    train_loss = total_loss / len(train_loader)
    dev_loss = evaluate_loss(model, dev_loader, DEVICE)

    print(
        f"\nEpoch {epoch + 1} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Dev Loss: {dev_loss:.4f}"
    )

    checkpoint_path = os.path.join(
        WORK_DIR,
        "checkpoints",
        f"epoch_{epoch + 1}.pt"
    )

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "config": config.to_dict()
        },
        checkpoint_path
    )

/tmp/ipykernel_58/1504707327.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(
/tmp/ipykernel_58/1504707327.py:20: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(


Epoch 1 | Step 0/14777 | Loss 8.6350
Epoch 1 | Step 100/14777 | Loss 6.6442
Epoch 1 | Step 200/14777 | Loss 5.9111
Epoch 1 | Step 300/14777 | Loss 5.6675
Epoch 1 | Step 400/14777 | Loss 5.5712
Epoch 1 | Step 500/14777 | Loss 5.6057
Epoch 1 | Step 600/14777 | Loss 5.2693
Epoch 1 | Step 700/14777 | Loss 5.0587
Epoch 1 | Step 800/14777 | Loss 5.1752
Epoch 1 | Step 900/14777 | Loss 5.2212
Epoch 1 | Step 1000/14777 | Loss 4.7572
Epoch 1 | Step 1100/14777 | Loss 4.9004
Epoch 1 | Step 1200/14777 | Loss 4.9435
Epoch 1 | Step 1300/14777 | Loss 4.4807
Epoch 1 | Step 1400/14777 | Loss 4.5429
Epoch 1 | Step 1500/14777 | Loss 4.6574
Epoch 1 | Step 1600/14777 | Loss 4.5274
Epoch 1 | Step 1700/14777 | Loss 4.5756
Epoch 1 | Step 1800/14777 | Loss 4.5800
Epoch 1 | Step 1900/14777 | Loss 4.6220
Epoch 1 | Step 2000/14777 | Loss 4.3372
Epoch 1 | Step 2100/14777 | Loss 4.5902
Epoch 1 | Step 2200/14777 | Loss 4.3164
Epoch 1 | Step 2300/14777 | Loss 4.5746
Epoch 1 | Step 2400/14777 | Loss 4.2832
Epoch 1 | St

In [31]:
# Cell 19 - Greedy generation

@torch.no_grad()
def generate_han(
    model,
    tokenizer,
    text,
    device,
    max_length=128
):
    model.eval()

    source_ids = tokenizer.encode_vi(
        text,
        add_special_tokens=True
    )

    source_ids = source_ids[:MAX_SOURCE_LENGTH]

    input_ids = torch.tensor(
        [source_ids],
        dtype=torch.long,
        device=device
    )

    attention_mask = (
        input_ids != tokenizer.vi_pad_id
    ).long()

    decoder_ids = torch.tensor(
        [[tokenizer.han_bos_id]],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_length):
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_ids
        )

        next_logits = outputs["logits"][:, -1, :]

        next_token = torch.argmax(
            next_logits,
            dim=-1,
            keepdim=True
        )

        decoder_ids = torch.cat(
            [decoder_ids, next_token],
            dim=1
        )

        if next_token.item() == tokenizer.han_eos_id:
            break

    return tokenizer.decode_han(
        decoder_ids[0].tolist()
    )

In [32]:
# Cell 20 - Inference test

model.eval()

examples = [
    "tôi đã học",
    "tôi thích học tiếng Trung",
    "Việt Nam"
]

for text in examples:
    prediction = generate_han(
        model,
        tokenizer,
        text,
        DEVICE
    )

    print("VI :", text)
    print("HAN:", prediction)
    print("-" * 50)

VI : tôi đã học
HAN: 我学习过
--------------------------------------------------
VI : tôi thích học tiếng Trung
HAN: 我喜欢中文学
--------------------------------------------------
VI : Việt Nam
HAN: 越南
--------------------------------------------------


In [33]:
# Cell 21 - Test generation on test set

model.eval()

NUM_TEST_SAMPLES = 20

for i, item in enumerate(test_data[:NUM_TEST_SAMPLES]):
    source_text = item["vi"]
    target_text = item["cn"]

    prediction = generate_han(
        model,
        tokenizer,
        source_text,
        DEVICE,
        max_length=MAX_TARGET_LENGTH
    )

    print(f"[{i + 1}]")
    print("VI    :", source_text)
    print("TARGET:", target_text)
    print("PRED  :", prediction)
    print("-" * 80)

[1]
VI    : " anh đã quan tâm tôi và thay đổi cuộc đời tôi . " anh ấy chính là Christopher , Tôi đã không khởi kiện Christopher .
TARGET: “ 你 关心 过 我 ， 改变 了 我 的 人生 。 ” 是 克里斯托弗 。 我 最终 没有 传讯 他 。
PRED  : “你对我的生活感兴趣，改变了我的生活。”他是弗里克托弗，我没有开始这个弗里克斯。
--------------------------------------------------------------------------------
[2]
VI    : " mày thấy tay giáo viên đằng kia không ?
TARGET: 「 看见 那 助教 没 ? 」 ？
PRED  : “你看到那边的老师吗？
--------------------------------------------------------------------------------
[3]
VI    : " người lạ mặt từ khắp đất nước muốn các em có chúng . " và bọn trẻ đáp , vô cùng hoài nghi ,
TARGET: 然后 他们 会 很 怀疑 的 说 ， “ 但是 它们 是 崭新 的 。 ”
PRED  : “陌生人想要他们的孩子。”孩子们，他们回答，无疑，
--------------------------------------------------------------------------------
[4]
VI    : " tôi là người mới . liệu tôi có thể mua chỉ một gram cần sa được không ? "
TARGET: “ 我 是 新来 的 。 你 能 卖 给 我 一克 的 大麻 吗 ？ ”
PRED  : “我是新人。我能买一克大麻的沙特？”
----------------------------------------------------------------------

In [34]:
# Cell 21 - Full test evaluation

model.eval()

predictions = []
references = []
sources = []

for i, item in enumerate(test_data):
    source_text = item["vi"]
    target_text = item["cn"]

    prediction = generate_han(
        model,
        tokenizer,
        source_text,
        DEVICE,
        max_length=MAX_TARGET_LENGTH
    )

    sources.append(source_text)
    references.append(target_text)
    predictions.append(prediction)

    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(test_data)}")

print("Finished.")
print("Total samples:", len(predictions))

Processed 100/1293
Processed 200/1293
Processed 300/1293
Processed 400/1293
Processed 500/1293
Processed 600/1293
Processed 700/1293
Processed 800/1293
Processed 900/1293
Processed 1000/1293
Processed 1100/1293
Processed 1200/1293
Finished.
Total samples: 1293


In [38]:
# Cell 22 - Test Metrics
#!pip install -q sacrebleu
import sacrebleu

char_predictions = [
    " ".join(list(pred))
    for pred in predictions
]

char_references = [
    " ".join(list(ref))
    for ref in references
]

char_bleu = sacrebleu.corpus_bleu(
    char_predictions,
    [char_references]
)

print("=" * 50)
print("CHARACTER-LEVEL BLEU")
print("=" * 50)
print(f"BLEU: {char_bleu.score:.2f}")
print("=" * 50)

CHARACTER-LEVEL BLEU
BLEU: 15.38


In [39]:
for i in range(20):
    print(f"[{i + 1}]")
    print("VI    :", sources[i])
    print("TARGET:", references[i])
    print("PRED  :", predictions[i])
    print("-" * 80)

[1]
VI    : " anh đã quan tâm tôi và thay đổi cuộc đời tôi . " anh ấy chính là Christopher , Tôi đã không khởi kiện Christopher .
TARGET: “ 你 关心 过 我 ， 改变 了 我 的 人生 。 ” 是 克里斯托弗 。 我 最终 没有 传讯 他 。
PRED  : “你对我的生活感兴趣，改变了我的生活。”他是弗里克托弗，我没有开始这个弗里克斯。
--------------------------------------------------------------------------------
[2]
VI    : " mày thấy tay giáo viên đằng kia không ?
TARGET: 「 看见 那 助教 没 ? 」 ？
PRED  : “你看到那边的老师吗？
--------------------------------------------------------------------------------
[3]
VI    : " người lạ mặt từ khắp đất nước muốn các em có chúng . " và bọn trẻ đáp , vô cùng hoài nghi ,
TARGET: 然后 他们 会 很 怀疑 的 说 ， “ 但是 它们 是 崭新 的 。 ”
PRED  : “陌生人想要他们的孩子。”孩子们，他们回答，无疑，
--------------------------------------------------------------------------------
[4]
VI    : " tôi là người mới . liệu tôi có thể mua chỉ một gram cần sa được không ? "
TARGET: “ 我 是 新来 的 。 你 能 卖 给 我 一克 的 大麻 吗 ？ ”
PRED  : “我是新人。我能买一克大麻的沙特？”
----------------------------------------------------------------------

In [ ]:
# Cell 22 - Save final model + tokenizer

FINAL_DIR = os.path.join(
    WORK_DIR,
    "final"
)

os.makedirs(FINAL_DIR, exist_ok=True)

torch.save(
    model.state_dict(),
    os.path.join(
        FINAL_DIR,
        "pytorch_model.bin"
    )
)

with open(
    os.path.join(FINAL_DIR, "config.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        config.to_dict(),
        f,
        ensure_ascii=False,
        indent=2
    )

with open(
    os.path.join(FINAL_DIR, "vocab_vi.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        vocab_vi,
        f,
        ensure_ascii=False,
        indent=2
    )

with open(
    os.path.join(FINAL_DIR, "vocab_han.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        vocab_han,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved to:", FINAL_DIR)